# Chapter 4 · Quantum Fourier Transform (QFT)

## Objectives

1. Understand the QFT as the quantum version of the classical DFT.
2. Build the QFT circuit for $n$ qubits.
3. Verify unitarity and compare with the analytical matrix.
4. Apply the QFT to a numerical example and visualize the result.

---

## 4.1 Definition

The Quantum Fourier Transform over $\mathbb{Z}_{N}$, with $N = 2^n$, is defined by:

$$\mathrm{QFT}|j\rangle = \frac{1}{\sqrt{N}} \sum_{k=0}^{N-1} e^{2\pi i jk/N} |k\rangle$$

Its efficient implementation requires only $O(n^2)$ gates (compared to $O(N \log N)$ for the classical FFT), with the circuit:

$$\mathrm{QFT}_n = (H \otimes I^{\otimes n-1}) \cdot R_2 \cdot R_3 \cdots R_n \cdots (H)_{\text{qubit } n} \cdot \mathrm{SWAP}$$

where $R_k$ is the H and phase-controlled CR$_k$ gate with phase $2\pi/2^k$.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector
from qiskit_aer import AerSimulator

from src.quantum_math import QuantumMath
from src.visualization import QuantumVisualization

print('Modules loaded.')

## 4.2 QFT Circuit

In [ ]:
def qft_circuit(n: int, inverse: bool = False, swap: bool = True) -> QuantumCircuit:
    """Builds the Quantum Fourier Transform circuit.

    Parameters
    ----------
    n : int
        Number of qubits.
    inverse : bool
        If True, builds the inverse QFT.
    swap : bool
        If True, adds final SWAP gates to reverse the qubit order.

    Returns
    -------
    QuantumCircuit
    """
    qc = QuantumCircuit(n, name='QFT' if not inverse else 'QFT†')

    def _qft_recursive(qc: QuantumCircuit, k: int):
        if k < 0:
            return
        qc.h(k)
        for j in range(k - 1, -1, -1):
            phase = 2 * np.pi / (2 ** (k - j + 1))
            qc.cp(phase, j, k)
        _qft_recursive(qc, k - 1)

    _qft_recursive(qc, n - 1)

    if swap:
        for i in range(n // 2):
            qc.swap(i, n - i - 1)

    if inverse:
        qc = qc.inverse()
        qc.name = 'QFT†'

    return qc

# Circuit for n=4
n = 4
qc_qft = qft_circuit(n)
print(f'QFT circuit for {n} qubits:')
print(qc_qft.draw('text'))
print(f'\nTotal operations: {qc_qft.size()}')

## 4.3 Numerical verification: comparison with the analytical matrix

In [ ]:
# Qiskit circuit matrix
n = 3
qc_qft_3 = qft_circuit(n)
U_qiskit = Operator(qc_qft_3).data

# Analytical matrix
U_analytic = QuantumMath.qft_matrix(n)

# Difference
diff = np.max(np.abs(U_qiskit - U_analytic))
print(f'Maximum difference |U_Qiskit - U_analytical| = {diff:.2e}')
print(f'Are they equal (tol=1e-10)? {np.allclose(U_qiskit, U_analytic, atol=1e-10)}')

# Matrix visualization
fig = QuantumVisualization.plot_unitary(
    U_analytic, title=f'QFT for {n} qubits'
)
plt.show()

## 4.4 Numerical example: QFT on a test state

In [ ]:
# Input state: |5〉 = |101〉 for n=3
n = 3
j_input = 5

# Prepare the state |j〉 as a vector of length 2^n
state_in = np.zeros(2**n, dtype=complex)
state_in[j_input] = 1.0
print(f'Input state: |{j_input}〉 = |{format(j_input, f"0{n}b")}〉')

# Apply analytical QFT
U_qft = QuantumMath.qft_matrix(n)
state_out = U_qft @ state_in

print(f'\nOutput state QFT|{j_input}〉:')
N = 2**n
for k, amp in enumerate(state_out):
    phase_exact = 2 * np.pi * j_input * k / N
    print(f'  |{format(k, f"0{n}b")}〉: {amp:.4f} '
          f'(phase = e^{{i·{phase_exact:.2f}}} = {np.exp(1j*phase_exact):.4f})')

# Visualization
fig = QuantumVisualization.plot_state_vector(state_out, title=f'QFT|{j_input}〉')
plt.show()

In [ ]:
# Verify QFT · QFT† = I
n = 4
qc_qft_n   = qft_circuit(n, inverse=False)
qc_iqft_n  = qft_circuit(n, inverse=True)

U_fwd = Operator(qc_qft_n).data
U_inv = Operator(qc_iqft_n).data

product = U_fwd @ U_inv
is_identity = np.allclose(product, np.eye(2**n), atol=1e-10)
print(f'QFT · QFT† = I for n={n}: {is_identity}')

## 4.5 Scalability: number of gates vs n

In [ ]:
n_list   = list(range(2, 14))
gates_qft = [qft_circuit(n).size() for n in n_list]
gates_th  = [n * (n + 1) // 2 + n // 2 for n in n_list]  # theoretical O(n²)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(n_list, gates_qft, 'o-', color='#58a6ff', linewidth=2, label='Qiskit (measured)')
ax.plot(n_list, gates_th,  's--', color='#f78166', linewidth=1.5, label='theoretical O(n²)')
ax.set_xlabel('Number of qubits (n)')
ax.set_ylabel('Number of gates')
ax.set_title('QFT circuit scalability')
ax.legend()
ax.grid(alpha=0.3)
ax.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 4.6 Proposed exercises

1. Apply the QFT to the state $|+\rangle^{\otimes 3}$ and determine the resulting state. What is its geometric interpretation?

2. Implement the inverse QFT without using Qiskit's `.inverse()`, manually reversing the gate order and negating the angles of the controlled rotations.

3. Demonstrate the **periodicity** property: if the input state is $\frac{1}{\sqrt{r}} \sum_{j=0}^{r-1} |j \cdot N/r\rangle$, the QFT produces another periodic state. Illustrate with $N=8$, $r=2$.

4. Investigate what happens to the QFT when the precision of the phases $R_k$ is limited to $k \leq k_{\max}$. How many steps are needed to maintain fidelity $> 0.99$?